## Compare pretrained backbones

In [1]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from configs.cfg import CFG
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *

if __name__ == "__main__":
    set_seed(CFG.SEED,CFG.DETERMINISTIC)
    df_wide = get_df()
    
    fold_indices = {}

    # sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    # splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=204)
    splitter = skf.split(X=df_wide, y=df_wide['biomass_bin'])

    for fold_id, (train_idx, val_idx) in enumerate(splitter):
        fold_indices[fold_id] = val_idx
        print(f"Fold {fold_id} captured: {len(val_idx)} images")

    # A. Define which folds go where
    train_folds = [1, 2, 3, 4]
    val_fold    = 0
    test_fold   = 4

    # B. Construct the final index lists
    # np.concatenate joins the arrays of indices together
    train_idx_final = np.concatenate([fold_indices[f] for f in train_folds])
    val_idx_final   = fold_indices[val_fold]
    test_idx_final  = fold_indices[test_fold]
    # C. Create the DataFrames
    train_df = df_wide.iloc[train_idx_final].reset_index(drop=True)
    val_df   = df_wide.iloc[val_idx_final].reset_index(drop=True)

    train_set = EmbeddingAugmentationDataset(train_idx_final, 'embeddings', n_aug=15, is_train=True)
    val_set   = EmbeddingAugmentationDataset(val_idx_final,   'embeddings', n_aug=15, is_train=False)

    # test_df  = df_wide.iloc[test_idx_final].reset_index(drop=True)
    # print(df_wide.head())
    print(f"Train Size: {len(train_df)}") # Should be roughly 60%
    print(f"Val Size:   {len(val_df)}")   # Should be roughly 20%
    # print(f"Test Size:  {len(test_df)}")  # Should be roughly 20%
    train_labels_tensor = torch.tensor(
        train_df[["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]].values, 
        dtype=torch.float32
    )
    # print(calculate_deltas(train_labels_tensor))
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    best_r2 = train_base(train_set,val_set, 0,group_name=group_name)
# Epoch 100 | TrainLoss 71.63072 | ValLoss 1655.10278 | ValR² -0.1487 (BEST)

/home/teo/repos/CSIRO---Image2Biomass-Prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
357 training images
Fold 0 captured: 72 images
Fold 1 captured: 72 images
Fold 2 captured: 71 images
Fold 3 captured: 71 images
Fold 4 captured: 71 images
Train Size: 285
Val Size:   72
Building model...


Epoch 01 | TrainLoss 1856.08766 | ValLoss 2081.85993 | ValR² -1.2450 (BEST) | GreenR² -0.82953 | DeadR² -1.02137 | CloverR² -0.20999 | GDMR² -1.43820 | TotalR² -2.31716
SAVED (R²: -1.2450)


Epoch 02 | TrainLoss 1208.09809 | ValLoss 768.30858 | ValR² 0.1715 (BEST) | GreenR² 0.03343 | DeadR² 0.06523 | CloverR² 0.21397 | GDMR² -0.05540 | TotalR² -0.09369
SAVED (R²: 0.1715)


Epoch 03 | TrainLoss 488.99301 | ValLoss 538.78600 | ValR² 0.4190 (BEST) | GreenR² 0.34239 | DeadR² -0.02909 | CloverR² 0.71436 | GDMR² 0.27878 | TotalR² 0.22357
SAVED (R²: 0.4190)


...

Epoch 35 | TrainLoss 57.44640 | ValLoss 207.95695 | ValR² 0.7757 (BEST) | GreenR² 0.77223 | DeadR² 0.23783 | CloverR² 0.87557 | GDMR² 0.72025 | TotalR² 0.70759
SAVED (R²: 0.7757)


Epoch 36 | TrainLoss 61.36519 | ValLoss 203.12686 | ValR² 0.7810 (BEST) | GreenR² 0.78796 | DeadR² 0.30889 | CloverR² 0.87657 | GDMR² 0.72286 | TotalR² 0.71215
SAVED (R²: 0.7810)


Epoch 37 | TrainLoss 56.33478 | ValLoss 199.57850 | ValR² 0.7848 (BEST) | GreenR² 0.76810 | DeadR² 0.38344 | CloverR² 0.84971 | GDMR² 0.71054 | TotalR² 0.72700
SAVED (R²: 0.7848)


train:   0%|          | 0/36 [00:00<?, ?it/s]         Exception ignored in: <function _releaseLock at 0x7f60997cb9a0>
Traceback (most recent call last):
  File "/usr/lib/python3.10/logging/__init__.py", line 228, in _releaseLock
    def _releaseLock():
KeyboardInterrupt: 


RuntimeError: DataLoader worker (pid(s) 23537, 23538, 23539) exited unexpectedly

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()
del model, train_loader, val_loader, optimizer, main_scheduler

## Cross Validation Training

In [ ]:
import pandas as pd
import gc
from datetime import datetime
from sklearn.model_selection import KFold, StratifiedGroupKFold
from configs.cfg import CFG
from torch.amp import GradScaler
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *
from utils.utils import *

if __name__ == "__main__":
    set_seed(13, CFG.DETERMINISTIC)
    df_wide = get_df()
    # best_seed = find_best_seed(df_wide, 1000)
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=13)
    kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=13)
    splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])
    spl = kf.split(X=df_wide, y=df_wide)
    # folds_list = list(splitter)

    check_splits(spl, df_wide)
    # Store the best R2 score from each fold
    all_fold_best_scores = []
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    for fold, (tr_idx, val_idx) in enumerate(splitter):
        print('\n' + '='*70)
        print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
        print('='*70)
        fold_dir_512 = os.path.join('cached_folds_date_state_20_512', f"fold_{fold+1}")
        fold_dir_768 = os.path.join('cached_folds_date_state_20', f"fold_{fold+1}")
        fold_dir_1008 = os.path.join('cached_folds_date_state_20_1008', f"fold_{fold+1}")

        tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
        val_df = df_wide.iloc[val_idx].reset_index(drop=True)

        # best_r2 = train_base(fold_dir_768,tr_df,val_df,fold,group_name = group_name)
        # best_r2 = train_tree(fold_dir_768, fold, verbose=True)
        # best_r2 = train_knn(fold_dir_768, fold, verbose=True)
        
        all_fold_best_scores.append(best_r2)

    # Cleanup
    torch.cuda.empty_cache()
    gc.collect()
    # all_fold_sizes = [83,81,66,71,56]# Change by hand if folds change date
    all_fold_sizes = [76,74,90,57,60]# Change by hand if folds change state+date
    weighted_cv_score = np.average(all_fold_best_scores, weights=all_fold_sizes)

    # 2. Standard Deviation
    # For std, a simple np.std is usually fine to just show "stability," 
    # but you can stick to the simple calculation for this.
    std_cv_score = np.std(all_fold_best_scores)
    print('\n' + '='*70)
    print('         FINAL CROSS-VALIDATION SCORE')
    print('='*70)
    print(f'Public LB Score: 0.58')
    # Notice we use the weighted score here
    print(f'Local CV Score: {weighted_cv_score:.4f} ± {std_cv_score:.4f}')
    print('\nIndividual Fold Scores:')
    for i, (score, size) in enumerate(zip(all_fold_best_scores, all_fold_sizes)):
        print(f'  Fold {i+1} (n={size}): {score:.4f}')
    # Local CV Score: 0.8078 ± 0.0415
    # 0.7953 ± 0.0358 - mse
    # 512,mse: 0.7993 ± 0.0364
    # 768,mse: 0.7933 ± 0.0370
    # 1008,mse: 0.7899 ± 0.0413


/home/teo/repos/CSIRO---Image2Biomass-Prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
357 training images
1     | 285          | 72         | 20.17 % | Dry_Total_g:48.9660  | Dry_Green_g:29.7216  | Dry_Dead_g:11.0067  | Dry_Clover_g:8.2377   | GDM_g:37.9594   | Weighted_g:36.9715 
2     | 285          | 72         | 20.17 % | Dry_Total_g:49.1368  | Dry_Green_g:30.2206  | Dry_Dead_g:13.2839  | Dry_Clover_g:5.6324   | GDM_g:35.8529   | Weighted_g:36.6527 
3     | 286          | 71         | 19.89 % | Dry_Total_g:37.5755  | Dry_Green_g:21.3431  | Dry_Dead_g:10.4752  | Dry_Clover_g:5.7616   | GDM_g:27.1046   | Weighted_g:27.9666 
4     | 286          | 71         | 19.89 % | Dry_Total_g:40.3740  | Dry_Green_g:21.4395  | Dry_Dead_g:11.7889  | Dry_Clover_g:7.1457   | GDM_g:28.5851   | Weighted_g:29.9414 
5     | 286          | 71         | 19.89 % | Dry_Total_g:50.4330  | Dry_Green_g:30.3046  | Dry_Dead_g:13.6653  | Dry_Clover_g:6.4631   | GDM_g:36.7677   | Weighted_g:37.6134 
-----------------------------------------------------------------
Max deviation from

NameError: name 'best_r2' is not defined

In [2]:
import torch
clean = torch.load("results/experiment_2/full_results.pt", weights_only=False)

In [3]:
print(clean)

{'leave_one_period_out': {'seed_results': [{'weighted_r2': np.float64(0.7284375630959824), 'std_weighted_r2': np.float64(0.02244528811245169), 'per_rmse': array([15.094185, 10.580616,  9.030105, 13.865666, 14.890785],
      dtype=float32), 'std_per_rmse': array([6.2543726, 3.2789497, 0.8905008, 3.3863742, 3.845223 ],
      dtype=float32), 'per_mae': array([10.187103 ,  7.9853706,  4.723753 ,  9.931687 , 10.435615 ],
      dtype=float32), 'std_per_mae': array([3.9725876 , 2.5968966 , 0.48603868, 2.0878992 , 2.1385243 ],
      dtype=float32), 'per_bias': array([-2.0207536, -2.6923752, -0.3002886,  0.2319889, -2.4596515],
      dtype=float32), 'std_per_bias': array([2.2997496, 3.8148248, 3.2738633, 4.164307 , 2.1611803],
      dtype=float32), 'per_target_r2': {'Dry_Green_g': np.float32(0.59729594), 'Dry_Dead_g': np.float32(0.036765695), 'Dry_Clover_g': np.float32(-2.9581492), 'GDM_g': np.float32(0.6454326), 'Dry_Total_g': np.float32(0.6569186)}, 'std_per_target_r2': {'Dry_Green_g': np.flo